In [3]:
import re

Q. 1. Match a pair of words delimited by an equal sign and swap those words.

In [5]:
reobj = re.compile(r"(\w+)=(\w+)")
subject = "100=marks"
result = reobj.sub(r"\2=\1",subject)
result

# this will also work

# result  = re.sub(r"(\w+)=(\w+)",r"\2=\1|,subject) will also work


'marks=100'

Q.2. Re-attempt the above problem using named backreferences

In [4]:
reobj = re.compile(r"(?P<left>\w+)=(?P<right>\w+)")
subject = "100=marks"
result = reobj.sub(r"\g<right>=\g<left>",subject)
result


'marks=100'

Q.3 Suppose you have an HTML file in which various passages are marked as bold with <b> tags. Between each pair of bold tags, you want to replace all matches of the regular expression "before" with the replacement text "After".

Example: When processing the string : 

"before <b>first before</b> before <b>before before</b>", you want to end up with : 

"before <b>first after</b> before <b> after after</b>"

# Incorrect Solution

In [20]:
import re
html_string = "before <b>first before before</b> before <b>before before</b>"

pattern1 = r"(?s).*(?=.*\s<b>.*)(before).*(?=</b>.*)"

pattern2 = r"(?s).*?(?=.*?\s<b>.*?)(before).*?(?=</b>.*)"

print(re.findall(pattern1,html_string))

print(re.findall(pattern2,html_string))

['before']
['before', 'before']


What regex actually does ?  Regex does NOT enumerate “everything possible”. It performs: leftmost-first, non-overlapping search.

**Mechanism**

At any position:

1. Try to match the pattern

2. If success → record match

3. Move pointer to end of that match

4. Repeat

In [11]:
import re

text = "aaaa"
print(re.findall(r"aa", text))

['aa', 'aa']


The output is Not: ['aa', 'aa', 'aa']

Why: Because Matches are non-overlapping

**Overlapping requires explicit construction**

In [12]:
import re

text = "aaaa"
print(re.findall(r"(?=(aa))", text))

['aa', 'aa', 'aa']


**Backtracking misconception**

Regex does explore alternatives, but: only to find ONE valid match at a given position. It does NOT: return all possible matches at that position

# Case 1

In [ ]:
"""
(?s).*(?=.*\s<b>.*)(before).*(?=</b>.*)
"""

In [ ]:

"""
**Structure**

.*                     → greedy prefix - consume everything first.

(?=.*\s\<b>.*)          → assert: there exists " \<b>" somewhere ahead

(before)               → capture target

.*                     → greedy suffix

(?=\</b>.*)             → assert: there exists "\</b>" somewhere ahead

**Execution model**

**Step 1 — Greedy prefix**

.*  → consumes the entire string initially

Pointer moves to end.

**Step 2 — Backtracking**

Engine backtracks to allow:

(before) + (?=\</b>.*)

to succeed.

**Step 3 — First satisfiable configuration**

Engine finds the rightmost "before" such that:

there exists \</b> somewhere after it

**Result**

Only ONE match is produced

Captured value:

typically the last "before" inside the last \<b>...\</b>

"""

# Case 2

In [ ]:
"""
(?s).*?(?=.*?\s<b>.*)(before).*?(?=</b>.*)
"""

In [ ]:
"""
.*?                    → lazy prefix
(?=.*?\s<b>.*)         → assert: there exists " <b>" ahead
(before)               → capture
.*?                    → lazy suffix
(?=</b>.*)             → assert: there exists "</b>" ahead
"""

In [ ]:
"""
Execution model

Step 1 — Lazy prefix

.*? → start with smallest possible prefix (empty)

So engine tries matching at every position.

Step 2 — At each position

Try:

(before)

AND

(?=</b>.*)

Key condition

(?=</b>.*)

means:

there exists a closing </b> somewhere in the future

What matches

Any "before" satisfying:

∃ </b> somewhere later in the string

This includes:

- "before" outside <b>

- "before" inside <b>

Result

['before', 'before' ]

(all qualifying occurrences)

"""

In [ ]:
"""
What the problem actually requires

For a word "before" to be valid:

It must lie INSIDE a <b> ... </b> pair

Formally:

∃ "<b>" before the word

AND

∃ "</b>" after the word

AND

the nearest "</b>" after "<b>" must be after the word

This is a bounded interval constraint.

What your patterns actually enforce

Both patterns enforce only:

∃ "</b>" somewhere AFTER the word

and

∃ "<b>" somewhere in the string (via lookahead)

Why that is insufficient

Consider this word:

before <b>first before before</b> before <b>before before</b>

       ↑
       
       this "before"

This "before" is OUTSIDE <b>...</b>

But your condition:

(?=</b>.*)

is TRUE because:

there exists a </b> later in the string

So your regex accepts invalid cases.


"""

# Correct solution

In [6]:
import re

html_string = "before <b>first before before</b> before <b>before before</b>"

inner_re = re.compile("before")

def replace_within(matchobj):
    return inner_re.sub("after",matchobj.group())

result = re.sub("<b>.*?</b>",replace_within,html_string)

print(result)

before <b>first after after</b> before <b>after after</b>


In [ ]:
""" 
Apply to your string
before <b>first before before</b> before <b>before before</b>

Iteration 1

Match:

<b>first before before</b>
Build result
append text before match:
"before "

append replacement:
"<b>first after after</b>"

So far:

before <b>first after after</b>

Pointer moves to end of first match.

Iteration 2

Next match:

<b>before before</b>
Append middle text
" before "
Append replacement
"<b>after after</b>"

Now:

before <b>first after after</b> before <b>after after</b>
End

No more matches → append remaining text (none here)

Key realization

No “joining” step exists

The string is constructed piece-by-piece during matching
Visual breakdown
[before ] 
+ [<b>first after after</b>] 
+ [ before ] 
+ [<b>after after</b>]
Why this works cleanly
Matches never overlap
→ each replacement occupies a distinct segment
→ assembly is deterministic
Final invariant
re.sub builds the output in a streaming manner:
    prefix + replacement + prefix + replacement + ...

"""

Q.4. Positive lookahead — basic constraint

Match all words that are immediately followed by a digit

Example:

Input: abc1 def ghi2 xyz
Output: abc, ghi

In [10]:
import re
text = "abc1 def ghi24 xyz"
pattern1 = r"(\w+)(?=\d)" # ['abc', 'ghi2']
pattern2 = r"(?i)\b([a-z]+)\d+"

print(re.findall(pattern1, text))
print(re.findall(pattern2, text))

['abc', 'ghi2']
['abc', 'ghi']


Q5. Match all numbers NOT followed by "kg"

Example:

Input: 50kg 30m 100kg 25

Output: 30, 25

In [ ]:
import re
text = "50kg 30m 100kg 25"
pattern1 = r"\d+(?!kg)" 
pattern2 = r"\d+(?!\d{0,}kg)" # \d{0,} equivalent to \d*

print(re.findall(pattern1, text))# Doesnt work!
print(re.findall(pattern2, text))# ['30', '25'] Works!

['5', '30', '10', '25']
['30', '25']


Q.6 Match all digits that are preceded by a "$"

Example:

Input: $100 €200 $50

Output: 100, 50

In [17]:
import re
text = "$100 €200 $50"
pattern1 = r"(?<$)(\d+)" 
pattern2 = r"" # \d{0,} equivalent to \d*

print(re.findall(pattern1, text))# Doesnt work!
print(re.findall(pattern2, text))# ['30', '25'] Works!

PatternError: unknown extension ?<$ at position 1

Q.4. Suppose you have an HTML file in which you want to replace straight double quotes with single quotes but only the quotes outside of HTML tags. Quotes within HTML tags must remain the same. 


In [ ]:
"""
"text" <span class="middle">"text"</span> 

must turn into 

'text' <span class="middle">'text'</span>
"""

In [ ]:
import re
html_file = "\"text\" <span class=\"middle\">\"text\"</span>"
pattern = r"(?!<.*>)"
replace_what = re.compile("\"")

def func(match_obj):
    return replace_what.sub("'",match_obj)


result = re.sub(pattern,func,html_file)

print(result)


TypeError: expected string or bytes-like object, got 're.Match'